# 📨 Notebook 1: Writes Get Lost When a Replica Is Down

**The setup:** A 3-replica system. The coordinator forwards each write to all three replicas. One replica is temporarily down.

Without **hinted handoff**, the down replica simply *misses* every write that arrived while it was away.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/hinted-handoff
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: forward-and-forget

In [ ]:
from dataclasses import dataclass, field
from typing import Dict

@dataclass
class Replica:
    name: str
    up: bool = True
    data: Dict[str, str] = field(default_factory=dict)

    def write(self, key, value):
        if not self.up:
            print(f'  ⚠ {self.name} is DOWN — write dropped')
            return False
        self.data[key] = value
        return True

replicas = [Replica('r1'), Replica('r2'), Replica('r3')]

def coordinator_write(key, value):
    for r in replicas:
        r.write(key, value)

replicas[2].up = False  # r3 dies
for i in range(5):
    coordinator_write(f'k{i}', f'v{i}')

replicas[2].up = True  # r3 comes back online
for r in replicas:
    print(f'{r.name}: {r.data}')


When `r3` recovers, it has **none** of the writes that arrived during its outage. Anti-entropy repair will *eventually* fix this — but until then, reads from `r3` see stale data.

👉 Next: have the coordinator **save a hint** for each write meant for a downed node and replay it on recovery.